In [2]:
import sqlite3
import pandas as pd

In [3]:
# ----------------------------------------------------
# 1. Database Connection & Schema Setup
# ----------------------------------------------------
DB_NAME = "proact.db"
conn = sqlite3.connect(DB_NAME)
cursor = conn.cursor()

In [4]:
# Enforce foreign keys in SQLite
cursor.execute("PRAGMA foreign_keys = ON;")

## Add Course tables to SQLite DB (These are tables with pre-loaded course and career information stored in local csv files)

In [2]:
# Define table schemas explicitly to preserve data types and primary keys
cursor.executescript("""
CREATE TABLE IF NOT EXISTS careers (
    career_id INTEGER PRIMARY KEY,
    career_name VARCHAR(150) UNIQUE NOT NULL,
    industry VARCHAR(100) NOT NULL,
    description TEXT NULL,
    created_at DATETIME NOT NULL
);

CREATE TABLE IF NOT EXISTS courses (
    course_id INTEGER PRIMARY KEY,
    sem1_course_code FLOAT UNIQUE,
    sem2_course_code FLOAT UNIQUE,
    course_name VARCHAR(150) NOT NULL,
    department VARCHAR(100) NOT NULL,
    description TEXT NULL,
    credits INTEGER,
    course_level INTEGER NULL,
    pathway BOOLEAN NULL,
    is_ap BOOLEAN NULL,
    is_honors BOOLEAN NULL,
    is_dual_enrollment BOOLEAN NULL,
    created_at DATETIME NOT NULL,
    updated_at DATETIME NULL
);

CREATE TABLE IF NOT EXISTS pathways (
    pathway_id INTEGER PRIMARY KEY AUTOINCREMENT,
    pathway_name TEXT NOT NULL,
    description TEXT NULL,
    created_at DATETIME NOT NULL
);

CREATE TABLE IF NOT EXISTS prerequisite_groups (
    prerequisite_group_id TEXT PRIMARY KEY,
    course_id INTEGER NOT NULL,
    parent_group_id TEXT NULL,
    group_operator BOOLEAN NULL,
    group_order INTEGER NOT NULL,
    source_expression TEXT NULL,
    depth INTEGER NOT NULL,
    created_at DATETIME NOT NULL
);


CREATE TABLE IF NOT EXISTS prerequisites (
    prerequisite_id TEXT PRIMARY KEY,
    prerequisite_group_id TEXT NOT NULL,
    requirement_order INTEGER NOT NULL,
    prerequisite_course_id TEXT,
    requirement_type TEXT NOT NULL,
    requirement_text TEXT,
    source_requirement TEXT,
    resolution_status TEXT NOT NULL,

    FOREIGN KEY (prerequisite_group_id)
        REFERENCES prerequisite_groups (prerequisite_group_id)
        ON DELETE CASCADE,

    FOREIGN KEY (prerequisite_course_id)
        REFERENCES courses (course_id)
        ON DELETE CASCADE
);

""")
conn.commit()

# Create application tables in SQLite DB (These are tables that will be populated by the application inputs)

In [5]:
# Define table schemas explicitly to preserve data types and primary keys
cursor.executescript("""
-- =====================================================
-- 1. DEPARTMENTS
-- Academic departments that offer courses
-- =====================================================
CREATE TABLE IF NOT EXISTS departments (
    department_id INTEGER PRIMARY KEY AUTOINCREMENT,
    department_name TEXT NOT NULL UNIQUE,
    description TEXT,
    created_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP
);

-- =====================================================
-- 2. USERS
-- Login identity and authentication information
-- =====================================================
CREATE TABLE IF NOT EXISTS users (
    user_id INTEGER PRIMARY KEY AUTOINCREMENT,
    email TEXT NOT NULL UNIQUE COLLATE NOCASE,
    password_hash TEXT NOT NULL,
    role TEXT NOT NULL DEFAULT 'student'
        CHECK (role IN ('student', 'parent', 'counselor', 'admin')),
    is_active INTEGER NOT NULL DEFAULT 1
        CHECK (is_active IN (0, 1)),
    created_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,
    updated_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP
);

-- =====================================================
-- 3. STUDENTS
-- Student-specific profile
-- =====================================================
CREATE TABLE IF NOT EXISTS students (
    student_id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id INTEGER NOT NULL UNIQUE,
    first_name TEXT NOT NULL,
    last_name TEXT NOT NULL,
    school_name TEXT NOT NULL DEFAULT 'Langston High School',
    current_grade INTEGER NOT NULL
        CHECK (current_grade BETWEEN 9 AND 12),
    graduation_year INTEGER NOT NULL,
    created_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,
    updated_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,

    FOREIGN KEY (user_id)
        REFERENCES users(user_id)
        ON DELETE CASCADE
);

-- =====================================================
-- 4. STUDENT INTERESTS
-- Academic, extracurricular, and career-related interests
-- =====================================================
CREATE TABLE IF NOT EXISTS student_interests (
    interest_id INTEGER PRIMARY KEY AUTOINCREMENT,
    student_id INTEGER NOT NULL,
    interest_category TEXT NOT NULL,
    interest_name TEXT NOT NULL,
    interest_level INTEGER
        CHECK (interest_level BETWEEN 1 AND 5),
    notes TEXT,
    created_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,

    FOREIGN KEY (student_id)
        REFERENCES students(student_id)
        ON DELETE CASCADE,

    UNIQUE (student_id, interest_category, interest_name)
);

-- =====================================================
-- 5. STUDENT GOALS
-- Supports both career and college goals
-- =====================================================
CREATE TABLE IF NOT EXISTS student_goals (
    goal_id INTEGER PRIMARY KEY AUTOINCREMENT,
    student_id INTEGER NOT NULL,
    goal_type TEXT NOT NULL
        CHECK (goal_type IN ('career', 'college')),
    goal_title TEXT NOT NULL,
    goal_description TEXT,
    career_id TEXT,
    institution_name TEXT,
    intended_major TEXT,
    degree_level TEXT,
    priority INTEGER NOT NULL DEFAULT 3
        CHECK (priority BETWEEN 1 AND 5),
    target_year INTEGER,
    status TEXT NOT NULL DEFAULT 'active'
        CHECK (status IN ('active', 'achieved', 'paused')),
    created_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,
    updated_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,

    FOREIGN KEY (student_id)
        REFERENCES students(student_id)
        ON DELETE CASCADE,

    FOREIGN KEY (career_id)
        REFERENCES careers(career_id)
        ON DELETE SET NULL
);

-- =====================================================
-- 6. STUDENT COURSES
-- Courses currently enrolled in or previously completed
-- =====================================================
CREATE TABLE IF NOT EXISTS student_courses (
    student_course_id INTEGER PRIMARY KEY AUTOINCREMENT,
    student_id INTEGER NOT NULL,
    course_id TEXT NOT NULL,
    school_year TEXT NOT NULL,
    grade_level INTEGER NOT NULL
        CHECK (grade_level BETWEEN 9 AND 12),
    term TEXT NOT NULL DEFAULT 'full_year'
        CHECK (term IN (
            'fall', 'spring', 'summer', 'full_year'
        )),
    status TEXT NOT NULL DEFAULT 'current'
        CHECK (status IN (
            'current', 'completed', 'withdrawn'
        )),
    final_grade TEXT,
    created_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,
    updated_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,

    FOREIGN KEY (student_id)
        REFERENCES students(student_id)
        ON DELETE CASCADE,

    FOREIGN KEY (course_id)
        REFERENCES courses(course_id)
        ON DELETE RESTRICT,

    UNIQUE (student_id, course_id, school_year, term)
);

-- =====================================================
-- 7. ACADEMIC PLANS
-- A student's saved four-year plan
-- =====================================================
CREATE TABLE IF NOT EXISTS academic_plans (
    plan_id INTEGER PRIMARY KEY AUTOINCREMENT,
    student_id INTEGER NOT NULL,
    plan_name TEXT NOT NULL DEFAULT 'My Four-Year Plan',
    starting_grade INTEGER NOT NULL DEFAULT 9
        CHECK (starting_grade BETWEEN 9 AND 12),
    graduation_year INTEGER NOT NULL,
    status TEXT NOT NULL DEFAULT 'draft'
        CHECK (status IN ('draft', 'finalized', 'archived')),
    created_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,
    updated_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,

    FOREIGN KEY (student_id)
        REFERENCES students(student_id)
        ON DELETE CASCADE
);

-- =====================================================
-- 8. ACADEMIC PLAN COURSES
-- Courses assigned to a specific year in a plan
-- =====================================================
CREATE TABLE IF NOT EXISTS academic_plan_courses (
    plan_course_id INTEGER PRIMARY KEY AUTOINCREMENT,
    plan_id INTEGER NOT NULL,
    course_id TEXT NOT NULL,
    planned_grade INTEGER NOT NULL
        CHECK (planned_grade BETWEEN 9 AND 12),
    school_year TEXT,
    term TEXT NOT NULL DEFAULT 'full_year'
        CHECK (term IN (
            'fall', 'spring', 'summer', 'full_year'
        )),
    requirement_category TEXT NOT NULL DEFAULT 'elective'
        CHECK (requirement_category IN (
            'core', 'pathway', 'elective', 'other'
        )),
    status TEXT NOT NULL DEFAULT 'planned'
        CHECK (status IN (
            'planned', 'approved', 'completed', 'removed'
        )),
    notes TEXT,
    created_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,

    FOREIGN KEY (plan_id)
        REFERENCES academic_plans(plan_id)
        ON DELETE CASCADE,

    FOREIGN KEY (course_id)
        REFERENCES courses(course_id)
        ON DELETE RESTRICT
);

-- =====================================================
-- INDEXES
-- Improve common student and planning queries
-- =====================================================
CREATE INDEX IF NOT EXISTS idx_students_user
    ON students(user_id);

CREATE INDEX IF NOT EXISTS idx_interests_student
    ON student_interests(student_id);

CREATE INDEX IF NOT EXISTS idx_goals_student
    ON student_goals(student_id);

CREATE INDEX IF NOT EXISTS idx_student_courses_student
    ON student_courses(student_id);

CREATE INDEX IF NOT EXISTS idx_plan_student
    ON academic_plans(student_id);

CREATE INDEX IF NOT EXISTS idx_plan_courses_plan
    ON academic_plan_courses(plan_id);

CREATE INDEX IF NOT EXISTS idx_plan_courses_course
    ON academic_plan_courses(course_id);

""")
conn.commit()

In [3]:
# ----------------------------------------------------
# 2. Data Cleaning Function
# ----------------------------------------------------
def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """Standardizes string values, strips whitespace, and removes blank duplicates."""
    # Strip whitespace from column headers
    df.columns = df.columns.str.strip()

    # Strip leading/trailing whitespaces from string columns
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].astype(str).str.strip()
        # Convert placeholder string blanks back to actual nulls
        df[col] = df[col].replace({"": None, "nan": None, "None": None})

    # Drop fully duplicate records
    df = df.drop_duplicates()
    return df

In [4]:
# ----------------------------------------------------
# 3. Load, Clean, and Push to Database
# ----------------------------------------------------
def push_csv_to_sqlite(csv_path: str, table_name: str, connection: sqlite3.Connection):
    try:
        # Load CSV
        df = pd.read_csv(csv_path)
        
        # Clean dataframe
        df_cleaned = clean_dataframe(df)

        # Write to SQLite
        # 'append' preserves the CREATE TABLE schema and types defined above
        df_cleaned.to_sql(
            name=table_name,
            con=connection,
            if_exists="append",
            index=False
        )
        print(f"Successfully loaded {len(df_cleaned)} rows into '{table_name}'.")

    except FileNotFoundError:
        print(f"Error: Could not find '{csv_path}'.")
    except sqlite3.IntegrityError as e:
        print(f"Integrity Error writing to '{table_name}': {e}")
    except Exception as e:
        print(f"Failed to process '{table_name}': {e}")

In [5]:
# Example usages:
push_csv_to_sqlite("careers.csv", "careers", conn)
push_csv_to_sqlite("courses.csv", "courses", conn)
push_csv_to_sqlite("pathways.csv", "pathways", conn)
push_csv_to_sqlite("prerequisite_groups.csv", "prerequisite_groups", conn)
push_csv_to_sqlite("prerequisites.csv", "prerequisites", conn)

Successfully loaded 25 rows into 'careers'.
Successfully loaded 154 rows into 'courses'.
Successfully loaded 13 rows into 'pathways'.
Successfully loaded 119 rows into 'prerequisite_groups'.
Successfully loaded 165 rows into 'prerequisites'.


In [7]:
# ----------------------------------------------------
# 4. Verify Database Records
# ----------------------------------------------------
def inspect_table(table_name: str, connection: sqlite3.Connection, limit: int = 5):
    query = f"SELECT * FROM {table_name} LIMIT {limit};"
    df_check = pd.read_sql_query(query, connection)
    print(f"\nSample data from '{table_name}':")
    print(df_check)

In [10]:
inspect_table("careers", conn)

# Close connection when complete
#conn.close()


Sample data from 'careers':
   career_id                               career_name  \
0          1                          Registered Nurse   
1          2                         Medical Assistant   
2          3                       Pharmacy Technician   
3          4  Medical & Clinical Laboratory Technician   
4          5                        Software Developer   

                 industry                                        description  \
0              Healthcare  Provides patient care using clinical judgment,...   
1              Healthcare  Supports clinical and administrative care with...   
2              Healthcare  Prepares medications and supports pharmacy ope...   
3              Healthcare  Performs laboratory testing using scientific r...   
4  Information Technology  Builds software using programming, logical rea...   

  created_at  
0  10/6/2026  
1  10/6/2026  
2  10/6/2026  
3  10/6/2026  
4  10/6/2026  
